# Lab 01 — SDLC Gatekeeper

## Objetivo

En este lab construiras un **evaluador automatizado de calidad de codigo** que actua como guardian (gatekeeper) en un pipeline CI/CD. Aprenderas tres estrategias complementarias de evaluacion:

| Estrategia | Cuando usarla | Coste |
|-----------|---------------|-------|
| **Exact Match** | Patrones literales prohibidos/requeridos | Gratuito, O(n) lineas |
| **Rule-Based (AST)** | Analisis estructural del codigo sin LLM | Gratuito, O(n) nodos |
| **LLM-as-Judge** | Comprension semantica o de intencion | Coste de API por llamada |

Al final del lab, el pipeline emite una decision binaria **GO / NO-GO** que puede bloquear un merge en GitHub Actions.

## ¿Que es un SDLC Gatekeeper?

Un **SDLC Gatekeeper** es un componente automatizado que evalua artefactos del ciclo de vida del software (codigo, configuraciones, dependencias) contra un conjunto de reglas de calidad antes de permitir que el cambio avance al siguiente estadio del pipeline.

### Analogia

Imagina la puerta de embarque de un aeropuerto: cada pasajero (pull request) debe pasar por una serie de controles (reglas). Si falla cualquier control, no embarca. El gatekeeper es ese proceso automatizado.

### ¿Por que tres tipos de evaluadores?

- **Exact Match**: es el control mas barato. `grep 'import requests'` detecta un import prohibido sin necesidad de entender el codigo. Deterministico, instantaneo, sin API.
- **Rule-Based (AST)**: algunas reglas requieren entender la *estructura* del codigo. Un magic number `30` es una violacion en `if age < 30`, pero no en `MAX_AGE = 30`. Solo el AST puede distinguir estos casos.
- **LLM-as-Judge**: las reglas semanticas son dificiles de formalizar. ¿Es `sk-prod-abc123secret` una API key hardcodeada? ¿Está esta clase haciendo demasiadas cosas? Un LLM entiende el contexto.

### Principio de diseno

> **Usa el evaluador mas simple que resuelva el problema.** No necesitas un LLM para detectar `import requests`. El LLM es el ultimo recurso, no el primero.

In [3]:
# Setup — imports y carga de API Key
import ast
import json
import os
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import anthropic
import yaml

try:
    from dotenv import load_dotenv
    load_dotenv()
    print("python-dotenv loaded")
except ImportError:
    print("python-dotenv not installed — using environment variables directly")

# Verify API key is available
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    api_key = input("Introduce tu ANTHROPIC_API_KEY: ").strip()
    os.environ["ANTHROPIC_API_KEY"] = api_key

print(f"API key configured: {'*' * 8}{api_key[-4:]}")

# Find repo root by walking up from cwd until config/rules.yaml is found
def _find_repo_root() -> Path:
    for p in [Path().resolve()] + list(Path().resolve().parents):
        if (p / "config" / "rules.yaml").exists():
            return p
    raise FileNotFoundError("Could not find repo root — run from inside the repository")

REPO_ROOT = _find_repo_root()
EXAMPLES_DIR = REPO_ROOT / "examples"
RULES_FILE = REPO_ROOT / "config" / "rules.yaml"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Rules file exists: {RULES_FILE.exists()}")

Repo root: /workspaces/sprint-3-lab01
Rules file exists: True


## Seccion 1 — Exact Match

El evaluador mas simple: busca patrones literales en el codigo fuente **linea a linea**.

### Cuando es suficiente
- Imports prohibidos: `import requests`
- Sentencias prohibidas: `print(`, `eval(`
- Patrones de namespace: `import *`

### Cuando NO es suficiente
- Cuando el contexto importa: `timeout=30` no es lo mismo que el literal `30` en una condicion
- Cuando necesitas entender la estructura: numero de argumentos de una funcion
- Cuando necesitas semantica: ¿este string parece una contrasena?

### Implementacion

El evaluador recibe una `rule` (diccionario del YAML) y el `source_code` como string. Devuelve un `EvalResult` con campos `passed`, `score` (0.0 o 1.0), y `evidence` (linea donde se encontro la violacion).

In [ ]:
# Implementacion ExactMatchEvaluator
@dataclass
class EvalResult:
    rule_id: str
    type: str
    passed: bool
    evidence: str
    score: float


class ExactMatchEvaluator:
    """Checks for forbidden or required literal patterns in source code."""

    def evaluate(self, rule: dict, source_code: str) -> EvalResult:
        pattern = rule["pattern"]
        match_type = rule.get("match_type", "forbidden")
        lines = source_code.splitlines()

        matches = [
            f"Line {i + 1}: {line.strip()}"
            for i, line in enumerate(lines)
            if pattern in line
        ]

        if match_type == "forbidden":
            passed = len(matches) == 0
            evidence = matches[0] if matches else "No violations found"
        else:  # required
            passed = len(matches) > 0
            evidence = matches[0] if matches else f"Required pattern '{pattern}' not found"

        return EvalResult(
            rule_id=rule["id"],
            type="exact_match",
            passed=passed,
            evidence=evidence,
            score=1.0 if passed else 0.0,
        )


exact_evaluator = ExactMatchEvaluator()
print("ExactMatchEvaluator definido")

In [ ]:
# Prueba ExactMatchEvaluator sobre api_client_bad.py
bad_api_client = (EXAMPLES_DIR / "bad_code" / "api_client_bad.py").read_text()
print("=== Contenido del archivo ===")
print(bad_api_client)

# Load only exact_match rules from YAML
with RULES_FILE.open() as f:
    all_rules = yaml.safe_load(f)["rules"]

exact_rules = [r for r in all_rules if r["type"] == "exact_match"]

print("\n=== Resultados Exact Match ===")
for rule in exact_rules:
    result = exact_evaluator.evaluate(rule, bad_api_client)
    status = "PASS" if result.passed else "FAIL"
    print(f"[{status}] {result.rule_id}: {result.evidence}")

## Seccion 2 — Rule-Based (AST)

El **Abstract Syntax Tree (AST)** de Python es una representacion en arbol de la estructura sintactica del codigo. La libreria `ast` de la stdlib permite recorrer ese arbol con el patron **Visitor**.

### ¿Por que AST y no regex?

Considera la regla `no_magic_numbers`:
- `MAX_AGE = 30` — constante nombrada, NO es una violacion
- `if age < 30` — literal en expresion condicional, SI es una violacion
- `timeout=30` — keyword argument con nombre explicito... depende del contexto

Un regex `\\b30\\b` no puede distinguir estos casos. El AST si, porque cada nodo tiene un tipo semantico: `Assign`, `Compare`, `keyword`.

### Nodos AST relevantes (Python 3.12)

```
ast.Constant   — literales numericos y de string (reemplaza ast.Num en 3.8+)
ast.Assign     — asignacion: MAX = 100
ast.AnnAssign  — asignacion anotada: MAX: int = 100
ast.Import     — import requests
ast.ImportFrom — from datetime import *
ast.ExceptHandler — except: / except Exception:
ast.FunctionDef    — def mi_funcion(a, b, c, d, e, f):
```

In [ ]:
# Implementacion RuleBasedEvaluator con visitores AST

class MagicNumberVisitor(ast.NodeVisitor):
    """Collects numeric literals that appear outside named constant assignments."""

    ALLOWED_VALUES = {0, 1, -1}

    def __init__(self):
        self.violations: list[tuple[int, float]] = []
        self._in_assignment = False

    def visit_Assign(self, node: ast.Assign):
        self._in_assignment = True
        self.generic_visit(node)
        self._in_assignment = False

    def visit_AnnAssign(self, node: ast.AnnAssign):
        self._in_assignment = True
        self.generic_visit(node)
        self._in_assignment = False

    def visit_Constant(self, node: ast.Constant):
        if (
            isinstance(node.value, (int, float))
            and not self._in_assignment
            and node.value not in self.ALLOWED_VALUES
        ):
            self.violations.append((node.lineno, node.value))


class ImportVisitor(ast.NodeVisitor):
    """Collects forbidden module imports."""

    def __init__(self, forbidden: list[str]):
        self.forbidden = forbidden
        self.violations: list[tuple[int, str]] = []

    def visit_Import(self, node: ast.Import):
        for alias in node.names:
            if any(alias.name.startswith(f) for f in self.forbidden):
                self.violations.append((node.lineno, alias.name))

    def visit_ImportFrom(self, node: ast.ImportFrom):
        if node.module and any(node.module.startswith(f) for f in self.forbidden):
            self.violations.append((node.lineno, node.module))


class BareExceptVisitor(ast.NodeVisitor):
    """Collects bare except clauses (no exception type specified)."""

    def __init__(self):
        self.violations: list[int] = []

    def visit_ExceptHandler(self, node: ast.ExceptHandler):
        if node.type is None:
            self.violations.append(node.lineno)
        self.generic_visit(node)


class FunctionArgVisitor(ast.NodeVisitor):
    """Collects functions that exceed the maximum allowed argument count."""

    def __init__(self, max_args: int):
        self.max_args = max_args
        self.violations: list[tuple[int, str, int]] = []

    def visit_FunctionDef(self, node: ast.FunctionDef):
        arg_count = len(node.args.args)
        if arg_count > self.max_args:
            self.violations.append((node.lineno, node.name, arg_count))
        self.generic_visit(node)

    visit_AsyncFunctionDef = visit_FunctionDef


class RuleBasedEvaluator:
    """Evaluates rules that require AST-level analysis."""

    def evaluate(self, rule: dict, source_code: str) -> EvalResult:
        try:
            tree = ast.parse(source_code)
        except SyntaxError as exc:
            return EvalResult(
                rule_id=rule["id"],
                type="rule_based",
                passed=False,
                evidence=f"Syntax error: cannot parse file — {exc}",
                score=0.0,
            )

        rule_id = rule["id"]
        if rule_id == "no_magic_numbers":
            return self._check_magic_numbers(rule, tree)
        elif rule_id == "use_internal_db":
            return self._check_forbidden_imports(rule, tree)
        elif rule_id == "no_bare_except":
            return self._check_bare_except(rule, tree)
        elif rule_id == "max_function_args":
            return self._check_function_args(rule, tree)
        return EvalResult(
            rule_id=rule_id, type="rule_based", passed=True,
            evidence=f"No handler for '{rule_id}' — skipped", score=1.0,
        )

    def _check_magic_numbers(self, rule, tree):
        visitor = MagicNumberVisitor()
        visitor.visit(tree)
        passed = len(visitor.violations) == 0
        if passed:
            evidence = "No magic numbers found"
        else:
            line, value = visitor.violations[0]
            evidence = f"Line {line}: magic number {value} outside named assignment"
        return EvalResult(rule_id=rule["id"], type="rule_based",
                          passed=passed, evidence=evidence, score=1.0 if passed else 0.0)

    def _check_forbidden_imports(self, rule, tree):
        forbidden = rule.get("forbidden_imports", [])
        visitor = ImportVisitor(forbidden)
        visitor.visit(tree)
        passed = len(visitor.violations) == 0
        if passed:
            evidence = "No forbidden imports found"
        else:
            line, name = visitor.violations[0]
            evidence = f"Line {line}: forbidden import '{name}'"
        return EvalResult(rule_id=rule["id"], type="rule_based",
                          passed=passed, evidence=evidence, score=1.0 if passed else 0.0)

    def _check_bare_except(self, rule, tree):
        visitor = BareExceptVisitor()
        visitor.visit(tree)
        passed = len(visitor.violations) == 0
        if passed:
            evidence = "No bare except clauses found"
        else:
            evidence = f"Line {visitor.violations[0]}: bare except — specify exception type"
        return EvalResult(rule_id=rule["id"], type="rule_based",
                          passed=passed, evidence=evidence, score=1.0 if passed else 0.0)

    def _check_function_args(self, rule, tree):
        max_args = rule.get("max_args", 5)
        visitor = FunctionArgVisitor(max_args)
        visitor.visit(tree)
        passed = len(visitor.violations) == 0
        if passed:
            evidence = f"All functions have {max_args} or fewer arguments"
        else:
            line, name, count = visitor.violations[0]
            evidence = f"Line {line}: '{name}' has {count} args (max {max_args})"
        return EvalResult(rule_id=rule["id"], type="rule_based",
                          passed=passed, evidence=evidence, score=1.0 if passed else 0.0)


rule_evaluator = RuleBasedEvaluator()
print("RuleBasedEvaluator definido")

In [ ]:
# Prueba RuleBasedEvaluator sobre data_processor_bad.py
bad_processor = (EXAMPLES_DIR / "bad_code" / "data_processor_bad.py").read_text()
print("=== Contenido del archivo ===")
print(bad_processor)

rule_based_rules = [r for r in all_rules if r["type"] == "rule_based"]

print("\n=== Resultados Rule-Based (AST) ===")
for rule in rule_based_rules:
    result = rule_evaluator.evaluate(rule, bad_processor)
    status = "PASS" if result.passed else "FAIL"
    print(f"[{status}] {result.rule_id}: {result.evidence}")

## Seccion 3 — LLM-as-Judge

Algunas reglas de calidad son **semanticas**: requieren entender el significado del codigo, no solo su estructura. Por ejemplo:

- `no_hardcoded_secrets`: ¿es `sk-prod-abc123secret` una API key? Un regex podria detectar el prefijo `sk-`, pero un LLM entiende el contexto completo.
- `single_responsibility`: ¿esta esta clase mezclando HTTP, logica de negocio y escritura en disco? Ningun visitor AST puede juzgar esto.
- `meaningful_naming`: ¿son los nombres descriptivos? Requiere comprension del lenguaje natural.

### Patron de evaluacion

Enviamos al LLM:
1. Un **system prompt** que le dice que actue como revisor estricto y devuelva unicamente JSON
2. Un **user message** con el criterio, el umbral de aprobacion, y el codigo fuente

El LLM devuelve: `{"score": 0.0-10.0, "passed": bool, "evidence": "..."}`

### Threshold

Cada regla llm_judge tiene un `score_threshold` en el YAML. Si `score >= threshold`, la regla pasa. Esto permite calibrar la estrictez por regla.

In [ ]:
# Implementacion LLMJudgeEvaluator
JUDGE_SYSTEM_PROMPT = """You are a strict code quality reviewer acting as an automated SDLC gate.
You will receive source code and a specific quality criterion to evaluate.
You must respond ONLY with a valid JSON object — no prose, no markdown.

Response schema:
{\"score\": <float 0.0-10.0>, \"passed\": <bool>, \"evidence\": \"<one concrete sentence citing line numbers>\"}

Scoring: 10.0 = zero violations, 0.0 = severe/multiple violations.
The 'passed' field is true when score >= the provided threshold."""


def _build_judge_prompt(rule: dict, source_code: str) -> str:
    return f"""Evaluate the following Python code against this criterion:

CRITERION: {rule['description']}
SCORE_THRESHOLD: {rule['score_threshold']}

DETAILS:
{rule['criteria']}

SOURCE CODE:
```python
{source_code}
```

Respond with the JSON schema only."""


class LLMJudgeEvaluator:
    """Evaluates rules using Claude as a semantic code reviewer."""

    def __init__(self, client: anthropic.Anthropic, model: str = "claude-sonnet-4-6"):
        self.client = client
        self.model = model

    def evaluate(self, rule: dict, source_code: str) -> EvalResult:
        prompt = _build_judge_prompt(rule, source_code)
        try:
            message = self.client.messages.create(
                model=self.model,
                max_tokens=256,
                system=JUDGE_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = message.content[0].text
        except anthropic.RateLimitError:
            print(f"Rate limit hit for rule {rule['id']} — waiting 30s...")
            time.sleep(30)
            message = self.client.messages.create(
                model=self.model,
                max_tokens=256,
                system=JUDGE_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = message.content[0].text
        except anthropic.APIError as exc:
            raise RuntimeError(f"Anthropic API error on rule {rule['id']}: {exc}") from exc

        try:
            parsed = json.loads(raw)
            score = float(parsed["score"])
            passed = bool(parsed["passed"])
            evidence = str(parsed["evidence"])
        except (json.JSONDecodeError, KeyError):
            return EvalResult(
                rule_id=rule["id"],
                type="llm_judge",
                passed=False,
                evidence=f"LLM returned non-JSON response: {raw[:100]}",
                score=0.0,
            )

        return EvalResult(
            rule_id=rule["id"],
            type="llm_judge",
            passed=passed,
            evidence=evidence,
            score=score,
        )


client = anthropic.Anthropic()
llm_evaluator = LLMJudgeEvaluator(client, model="claude-sonnet-4-6")
print("LLMJudgeEvaluator definido")

In [ ]:
# Prueba LLMJudgeEvaluator sobre service_bad.py
bad_service = (EXAMPLES_DIR / "bad_code" / "service_bad.py").read_text()
print("=== Contenido del archivo ===")
print(bad_service)

llm_rules = [r for r in all_rules if r["type"] == "llm_judge"]

print("\n=== Resultados LLM-as-Judge ===")
for rule in llm_rules:
    result = llm_evaluator.evaluate(rule, bad_service)
    status = "PASS" if result.passed else "FAIL"
    print(f"[{status}] {result.rule_id} (score={result.score:.1f}): {result.evidence}")

## Seccion 4 — Pipeline Completo

El `GatekeeperPipeline` orquesta los tres evaluadores en un unico flujo:

```
archivo.py
    |
    v
[Cargar rules.yaml]
    |
    +-- exact_match rules  -->  ExactMatchEvaluator
    +-- rule_based rules   -->  RuleBasedEvaluator (AST)
    +-- llm_judge rules    -->  LLMJudgeEvaluator  (Anthropic API)
    |
    v
  any(not passed)?
     SI  --> decision = "NO-GO"  -->  sys.exit(1)  [bloquea CI]
     NO  --> decision = "GO"     -->  sys.exit(0)  [CI continua]
```

### Logica AND estricta

La decision es **NO-GO** si **cualquier** regla falla. No hay ponderaciones ni mayorias. Esto es coherente con un gate real de CI/CD: un solo `import requests` bloquea el merge igual que una API key hardcodeada.

In [ ]:
# GatekeeperPipeline — orquesta los tres evaluadores
class GatekeeperPipeline:
    """Orchestrates all three evaluator types and produces a GO/NO-GO decision."""

    def __init__(self, rules: list[dict], model: str = "claude-sonnet-4-6"):
        self.rules = rules
        self.model = model
        self.exact_evaluator = ExactMatchEvaluator()
        self.rule_evaluator = RuleBasedEvaluator()
        self.llm_evaluator = LLMJudgeEvaluator(client, model)

    def run(self, file_path) -> dict:
        source_code = Path(file_path).read_text(encoding="utf-8")
        results: list[EvalResult] = []

        for rule in self.rules:
            rule_type = rule["type"]
            if rule_type == "exact_match":
                result = self.exact_evaluator.evaluate(rule, source_code)
            elif rule_type == "rule_based":
                result = self.rule_evaluator.evaluate(rule, source_code)
            elif rule_type == "llm_judge":
                result = self.llm_evaluator.evaluate(rule, source_code)
            else:
                result = EvalResult(
                    rule_id=rule["id"], type=rule_type, passed=True,
                    evidence=f"Unknown type '{rule_type}' — skipped", score=1.0,
                )
            results.append(result)

        failed = [r for r in results if not r.passed]
        passed_list = [r for r in results if r.passed]
        aggregate_score = sum(r.score for r in results) / len(results) if results else 0.0

        return {
            "run_id": datetime.utcnow().isoformat() + "Z",
            "file": str(file_path),
            "model": self.model,
            "results": [asdict(r) for r in results],
            "decision": "GO" if not failed else "NO-GO",
            "passed_rules": len(passed_list),
            "failed_rules": len(failed),
            "aggregate_score": round(aggregate_score, 2),
        }


pipeline = GatekeeperPipeline(rules=all_rules, model="claude-sonnet-4-6")
print("GatekeeperPipeline definido")

In [ ]:
# Ejecutar pipeline sobre todos los archivos de bad_code/
bad_code_files = list((EXAMPLES_DIR / "bad_code").glob("*.py"))
pipeline_reports = {}

for bad_file in sorted(bad_code_files):
    print(f"\nEvaluando: {bad_file.name}")
    report = pipeline.run(bad_file)
    pipeline_reports[bad_file.name] = report
    decision = report["decision"]
    print(f"  Decision: {decision} | Passed: {report['passed_rules']} | Failed: {report['failed_rules']} | Score: {report['aggregate_score']}")

In [ ]:
# Mostrar tabla de resultados detallada + decision GO / NO-GO
for filename, report in pipeline_reports.items():
    decision_label = report["decision"]
    print(f"\n{'='*60}")
    print(f"Archivo: {filename}  |  Decision: {decision_label}  |  Score: {report['aggregate_score']}")
    print(f"{'='*60}")
    print(f"{'Regla':<35} {'Tipo':<13} {'Estado':<6} {'Score':>5}")
    print(f"{'-'*35} {'-'*13} {'-'*6} {'-'*5}")
    for r in report["results"]:
        status = "PASS" if r["passed"] else "FAIL"
        print(f"{r['rule_id']:<35} {r['type']:<13} {status:<6} {r['score']:>5.1f}")

    failed_results = [r for r in report["results"] if not r["passed"]]
    if failed_results:
        print("\nViolaciones detectadas:")
        for r in failed_results:
            print(f"  - [{r['rule_id']}] {r['evidence']}")

In [ ]:
# Exportar resultado JSON al directorio results/ (formato compatible con GitHub Actions)
timestamp = datetime.utcnow().strftime("%Y-%m-%d_%H-%M")
model_id = "claude-sonnet-4-6"
output_filename = f"{timestamp}_lab01_{model_id}.json"
output_path = RESULTS_DIR / output_filename

# Aggregate all file reports into a single output
aggregate_output = {
    "run_id": datetime.utcnow().isoformat() + "Z",
    "lab": "01_sdlc_gatekeeper",
    "model": model_id,
    "files_evaluated": len(pipeline_reports),
    "reports": pipeline_reports,
    "overall_decision": "GO" if all(r["decision"] == "GO" for r in pipeline_reports.values()) else "NO-GO",
}

output_path.write_text(json.dumps(aggregate_output, indent=2), encoding="utf-8")
print(f"Resultado exportado a: {output_path}")
print(f"Decision global: {aggregate_output['overall_decision']}")

## Reflexion — ¿Cuando usar cada tipo de evaluador?

### Resumen de criterios de seleccion

| Evaluador | Usar cuando | No usar cuando |
|-----------|-------------|----------------|
| **Exact Match** | La presencia/ausencia de un token es condicion suficiente | El contexto cambia el significado del token |
| **Rule-Based (AST)** | Necesitas estructura sintatica: contar args, detectar contexto de asignacion, imports especificos | El criterio requiere entender la intencion o semantica del codigo |
| **LLM-as-Judge** | Reglas semanticas o de diseno: secretos, responsabilidad unica, nombrado | Patrones simples que un grep o AST resuelve — el LLM es mas caro y mas lento |

### Principio de costos

```
Exact Match:    O(1) por regla — costo: ~0
Rule-Based:     O(n) nodos AST — costo: ~0
LLM-as-Judge:   1 llamada API por regla — costo: ~$0.001-0.01 segun modelo y tamano del codigo
```

En un pipeline CI/CD que corre cientos de veces al dia, el coste de LLM-as-Judge se acumula. La estrategia optima es:

1. **Ejecutar primero Exact Match** — si ya hay violaciones, hacer fail-fast sin gastar en LLM
2. **Ejecutar Rule-Based** — filtrar mas violaciones estructurales
3. **Ejecutar LLM-as-Judge solo si los anteriores pasan** — o en un subset de reglas criticas

### Proximos pasos

- **Lab 02 — Architect Agent**: usa un agente con herramientas para tomar decisiones arquitectonicas mas complejas
- **Lab 03 — Performance**: optimiza el coste de LLM-as-Judge con **prompt caching** — comparte el codigo fuente entre llamadas
- **Lab 04 — Advanced**: combina evals con workflows de GitHub Actions para bloquear PRs automaticamente